# BuddyBook — เทรน sentiment v4 บน Colab (GPU)

ใช้ไฟล์ `buddybook_v4_colab.zip` (~420MB) ที่สร้างจากเครื่อง dev:
`apps/nlp-worker/train/colab/buddybook_v4_colab.zip` — ข้างในมีโมเดล v2, `data/train.csv`, `data/val.csv` และ `finetune.py`

**ก่อนเริ่ม**
1. อัปโหลด `buddybook_v4_colab.zip` ไว้ที่ Google Drive โฟลเดอร์บนสุด (My Drive)
2. เมนู **Runtime → Change runtime type → T4 GPU** แล้วกด Save
3. กด **Runtime → Run all** (จะมีหน้าต่างขออนุญาตเข้าถึง Google Drive ให้กดยืนยัน)

เสร็จแล้วจะได้ `buddybook_sentiment_v4.zip` ใน My Drive ดาวน์โหลดมาวางในเครื่องตามขั้นตอนท้ายสุด

In [ ]:
# 1) เช็คว่าได้ GPU — ถ้าขึ้น False ให้กลับไปเปลี่ยน runtime เป็น T4 GPU
import torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# 2) ติดตั้ง transformers เวอร์ชันเดียวกับ worker (4.44.0) — โมเดลที่ได้จะโหลดในเครื่องได้แน่นอน
!pip -q install transformers==4.44.0

In [ ]:
# 3) ดึงไฟล์จาก Google Drive แล้วแตก zip
from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/buddybook_v4_colab.zip /content/
!cd /content && rm -rf bb && mkdir bb && cd bb && unzip -q ../buddybook_v4_colab.zip && ls -R | head -20

In [ ]:
# 4) เทรนต่อจาก v2 — Wisesight ทั้งชุด ~15,500 แถว 3 epoch บน T4 ราว 15–25 นาที
# --synthetic-weight 8: คอมเมนต์นิยายที่แต่ง (211 แถว) จมในข้อมูลหมื่นกว่าแถว ทำซ้ำ 8 เท่าให้ได้ราว 10%
# เก็บ checkpoint ที่ macro-F1 บน val ดีที่สุดไว้ที่ model_v4 (ถ้าไม่ดีกว่า v2 จะได้ v2 เดิมกลับมา)
!cd /content/bb && python finetune.py --base model_v2 --data data --out model_v4 --epochs 3 --batch 32 --max-len 128 --synthetic-weight 8

In [ ]:
# 5) บีบอัดผลลัพธ์แล้วเก็บกลับไปที่ My Drive
!cd /content/bb && rm -f model_v4/training_args.bin && zip -q -r buddybook_sentiment_v4.zip model_v4 && ls -lh buddybook_sentiment_v4.zip
!cp /content/bb/buddybook_sentiment_v4.zip /content/drive/MyDrive/
print("เสร็จแล้ว — ไฟล์อยู่ที่ My Drive/buddybook_sentiment_v4.zip")

## หลังเทรนเสร็จ (ทำในเครื่อง)

1. ดาวน์โหลด `buddybook_sentiment_v4.zip` จาก My Drive
2. แตกไฟล์ จะได้โฟลเดอร์ `model_v4` — วางไว้ที่ `apps/nlp-worker/model_v4/` ในโปรเจกต์
   (ข้าง `model/` ที่ worker ใช้อยู่ · ทั้งสองโฟลเดอร์อยู่ใน .gitignore ไม่ขึ้น git)
3. บอก Claude ให้วัดผล v4 เทียบ v2 (`eval/evaluate.py` + `eval/calibrate.py`) — ถ้า v4 ดีกว่าจึงย้ายเข้า `model/` แทน v2

**อย่าเอา v4 ไปใช้เลยโดยไม่วัดผล** — macro-F1 บน val มาจากแหล่งเดียวกับข้อมูลเทรน ต้องดูผลบน `eval/test_set.csv` ด้วย